# ClaimLens — demo

> **Educational prototype — not insurance, legal or medical advice.** Synthetic claims only.

ClaimLens reads a health-insurance claim and its hospital bill, checks both against the policy wording
(IFFCO Tokio *Arogya Sanjeevani*, `ASJ-01`) and IRDAI rules, and returns **approve, partial, reject or
escalate**, with the payout and the clause behind every rupee deducted.

For each demo claim this notebook prints:
1. the claim text and bill,
2. every agent step as it runs (agent, tool, short result) — Policy Retrieval and Fraud Signal start together,
3. the final record: verdict, payout, deduction table, fraud score and reasons, cited clauses,
4. for scenario 4, the escalation, the reviewer's verdict and the closed record,
5. a summary of all seven scenarios, expected vs actual.

Runs write to `state/<claim_id>.json` and append to `logs/run_log.jsonl` (both git-ignored).

In [1]:
import json

from IPython.display import Markdown, display

from agents import human_desk
from tools.config import DISCLAIMER, project_path
from tools.decision_tools import format_inr
from tools.intake_tools import parse_bill
from tools.logger import read_log
from tools.schemas import ExpectedOutcome
from workflow import run_claim, summary

CLAIMS = project_path("claims")


def rupees(x):
    return "—" if x is None else f"₹{format_inr(x)}"


def table(headers, rows):
    out = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    out += ["| " + " | ".join(str(c) for c in row) + " |" for row in rows]
    return "\n".join(out)


def show_claim(cid):
    folder = CLAIMS / cid
    text = (folder / "claim.txt").read_text(encoding="utf-8")
    display(Markdown(f"### {cid} — claim message\n\n" + "\n".join("> " + ln for ln in text.splitlines())))
    if (folder / "bill.csv").exists():
        items = parse_bill(folder / "bill.csv")
        rows = [(i["desc"], i["category"], i["qty"], rupees(i["amount"])) for i in items]
        rows.append(("**Total**", "", "", f"**{rupees(sum(i['amount'] for i in items))}**"))
        display(Markdown("**Bill**\n\n" + table(["Item", "Category", "Qty", "Amount"], rows)))
    else:
        display(Markdown("**Bill:** none attached"))


def show_trace(cid):
    log = read_log(cid)
    start = max(i for i, e in enumerate(log) if e.get("tool") == "reset_state")
    run = [e for e in log[start + 1:] if e.get("tool") not in ("write_state", "set_status")]
    rows = [(e["ts"][11:23], e["agent"], e.get("tool", ""), e.get("summary", "").replace("|", "/")) for e in run]
    display(Markdown("**Agent trace** (from `logs/run_log.jsonl`)\n\n" + table(["Time (UTC)", "Agent", "Step / tool", "Result"], rows)))


def show_record(rec):
    s = summary(rec)
    if rec.get("halt"):
        display(Markdown(f"**Intake halted** — `{rec['halt']['reason']}`: {rec['halt']['message']}  \n"
                         f"Status: `{rec['status']}` · no downstream agent ran."))
        return
    d, f, c = rec["decision"], rec["fraud"], rec["cost"]
    card = (f"**Decision:** `{d['verdict']}` · payout **{rupees(s['payout'])}** of {rupees(rec['intake']['claimed'])} "
            f"claimed · confidence {d['confidence']} · status `{rec['status']}`")
    if d["escalation_reasons"]:
        card += f"  \n**Escalation:** {', '.join(d['escalation_reasons'])}"
    card += f"  \n**Rationale:** {d['rationale']}"
    card += f"  \n**Cited clauses:** {', '.join(f'`{x}`' for x in d['clauses']) or '— none'}"
    card += f"  \n**Fraud score:** {f['score']} {f['reasons'] or ''}"
    if f.get("waiting_period", {}).get("excluded"):
        w = f["waiting_period"]
        card += f"  \n**Waiting period:** {w['condition']} — excluded until {w['waiting_ends']} (`{w['clause_id']}`)"
    if c["deductions"]:
        rows = [(x["item"], rupees(x["amount"]), x["reason"], f"`{x['clause_id']}`" if x["clause_id"] else "—")
                for x in c["deductions"]]
        card += "\n\n**Deduction table**\n\n" + table(["Item", "Cut", "Reason", "Clause"], rows)
    if rec["human"].get("verdict"):
        h = rec["human"]
        card += (f"\n\n**Human review** {h['ticket']}: `{h['verdict']}` by {h['reviewer']} — payout "
                 f"{rupees(h['payout'])}. Note: {h['note']}")
    display(Markdown(card))


async def demo(cid):
    show_claim(cid)
    rec = await run_claim(cid)
    show_trace(cid)
    show_record(rec)
    return rec


print(DISCLAIMER)

Educational prototype — not insurance, legal or medical advice.


## Scenario 1 — clean approval
Appendectomy, ₹85,000 claimed, room + nursing at the ₹5,000/day cap, no risk flags. Expected: **approve**,
payout = claimed − IRDAI List I items − 5% co-pay.

In [2]:
rec1 = await demo("CLM-0001")

### CLM-0001 — claim message

> Claim for reimbursement under policy ASJ-01, policyholder PH-1001.
> 
> Patient: Ravi Kumar (synthetic). Admitted on 2 September 2026 to Sunrise Multispeciality
> Hospital (synthetic) with acute abdominal pain, diagnosed as acute appendicitis, and
> underwent laparoscopic appendectomy. Stayed in a twin-sharing room at Rs 4,000 per day (nursing Rs 1,000 per day)
> and was discharged on 5 September 2026.
> 
> Final hospital bill number SMH/26/04417. Total amount claimed: Rs 85,000.
> Discharge summary, final bill and investigation reports attached.

**Bill**

| Item | Category | Qty | Amount |
|---|---|---|---|
| Room charges (twin sharing) | room | 3 | ₹12,000 |
| Nursing charges | nursing | 3 | ₹3,000 |
| Surgeon fees - laparoscopic appendectomy | professional | 1 | ₹25,000 |
| Anaesthetist fees | professional | 1 | ₹8,000 |
| Operation theatre charges | ot | 1 | ₹15,000 |
| Medicines and IV fluids | pharmacy | 1 | ₹12,000 |
| Investigations (CBC, USG abdomen) | diagnostics | 1 | ₹8,800 |
| Gloves | consumables | 1 | ₹450 |
| Mask | consumables | 1 | ₹250 |
| Laundry charges | misc | 1 | ₹300 |
| Telephone charges | misc | 1 | ₹200 |
| **Total** |  |  | **₹85,000** |

**Agent trace** (from `logs/run_log.jsonl`)

| Time (UTC) | Agent | Step / tool | Result |
|---|---|---|---|
| 18:07:21.075 | workflow | start | intake |
| 18:07:21.080 | intake_agent | start | reading claim message and bill |
| 18:07:21.083 | intake_agent | extract_claim_fields | policy ASJ-01, claimed 85000, diagnosis: acute appendicitis; laparoscopic appendectomy |
| 18:07:22.137 | intake_agent | reconcile_fields | groq (qwen/qwen3.8-27b): 10 fields agree · groq: 884/900 requests, 84,743/90,000 tokens left today |
| 18:07:22.141 | intake_agent | validate_policy | ASJ-01 found in policies.json |
| 18:07:22.142 | intake_agent | parse_bill | 11 line items, total 85000 |
| 18:07:22.181 | workflow | fan-out | Policy Retrieval and Fraud Signal start together |
| 18:07:22.192 | policy_agent | start | limits + clauses for ASJ-01 |
| 18:07:22.193 | policy_agent | get_policy_limits | SI 500000, room cap 5000/day, co-pay 5% |
| 18:07:22.195 | fraud_agent | start | risk rules + prior-claim memory |
| 18:07:22.199 | fraud_agent | lookup_prior_claims | 0 earlier claims by PH-1001 |
| 18:07:22.205 | fraud_agent | check_waiting_period | no waiting period applies |
| 18:07:22.206 | fraud_agent | score_risk | score 0 [] |
| 18:07:22.238 | cost_agent | start | 11 bill lines against ASJ-01 limits |
| 18:07:22.247 | cost_agent | classify_items | not payable: ['Gloves', 'Mask', 'Laundry charges', 'Telephone charges'] |
| 18:07:22.247 | cost_agent | room_rent_deduction | cuts: none |
| 18:07:22.249 | cost_agent | compute_payable | payable 79610 of 85000; over-claim False |
| 18:07:23.261 | policy_agent | search_policy_clauses | 'room rent boarding nursing expenses per day limit' -> ['ASJ-01:4.1.i', 'ASJ-01:3.45', 'ASJ-01:4.1.1.note2'] |
| 18:07:23.263 | policy_agent | search_policy_clauses | 'intensive care unit ICU expenses per day limit' -> ['ASJ-01:4.1.ii', 'ASJ-01:3.26', 'ASJ-01:3.25'] |
| 18:07:23.263 | policy_agent | search_policy_clauses | 'rates exceeding limits proportion other expenses' -> ['ASJ-01:4.1.1.note2', 'ASJ-01:5'] |
| 18:07:23.264 | policy_agent | search_policy_clauses | 'co-payment admissible claim' -> ['ASJ-01:9.3', 'ASJ-01:3.11', 'ASJ-01:3.36.ii'] |
| 18:07:23.265 | policy_agent | search_policy_clauses | 'first thirty days waiting period illness' -> ['ASJ-01:6.2.i', 'ASJ-01:3.19', 'ASJ-01:6.3'] |
| 18:07:23.266 | policy_agent | search_policy_clauses | 'pre-existing disease waiting period' -> ['ASJ-01:6.1', 'ASJ-01:3.42', 'ASJ-01:3.44'] |
| 18:07:23.269 | policy_agent | search_irdai_rules | 'items for which coverage is not available' -> ['IRDAI:List-I'] |
| 18:07:23.269 | policy_agent | search_irdai_rules | 'items subsumed into room charges' -> ['IRDAI:List-II'] |
| 18:07:23.270 | policy_agent | search_irdai_rules | 'subsumed into procedure charges' -> ['IRDAI:List-III'] |
| 18:07:23.270 | policy_agent | search_irdai_rules | 'items subsumed into costs of treatment' -> ['IRDAI:List-IV'] |
| 18:07:23.270 | policy_agent | done | 23 clauses above the similarity threshold |
| 18:07:23.289 | workflow | fan-in | policy, fraud and cost signals in; Manager decides |
| 18:07:23.300 | manager_agent | start | fan-in: policy, fraud and cost signals |
| 18:07:23.302 | manager_agent | apply_decision_rules | approve payout 79610 |
| 18:07:23.302 | manager_agent | verify_citations | cited ['IRDAI:List-I', 'ASJ-01:9.3'] — all retrieved |
| 18:07:23.332 | manager_agent | write_decision | status closed |
| 18:07:23.332 | workflow | done | status closed |

**Decision:** `approve` · payout **₹79,610** of ₹85,000 claimed · confidence 1.0 · status `closed`  
**Rationale:** Approved: payable ₹79,610 of ₹85,000 claimed; deductions: Gloves ₹450 (IRDAI:List-I); Mask ₹250 (IRDAI:List-I); Laundry charges ₹300 (IRDAI:List-I); Telephone charges ₹200 (IRDAI:List-I); Co-payment 5% ₹4,190 (ASJ-01:9.3).  
**Cited clauses:** `IRDAI:List-I`, `ASJ-01:9.3`  
**Fraud score:** 0 

**Deduction table**

| Item | Cut | Reason | Clause |
|---|---|---|---|
| Gloves | ₹450 | non-payable item | `IRDAI:List-I` |
| Mask | ₹250 | non-payable item | `IRDAI:List-I` |
| Laundry charges | ₹300 | non-payable item | `IRDAI:List-I` |
| Telephone charges | ₹200 | non-payable item | `IRDAI:List-I` |
| Co-payment 5% | ₹4,190 | 5% co-payment on admissible claim | `ASJ-01:9.3` |

## Scenario 2 — partial: room-rent breach
Single AC room ₹7,000 + nursing ₹1,000 = ₹8,000/day against a ₹5,000 cap (`ASJ-01:4.1.i`). Every other expense
except medicines is cut in proportion 5,000 / 8,000 (`ASJ-01:4.1.1.note2`).

In [3]:
rec2 = await demo("CLM-0002")

### CLM-0002 — claim message

> Claim for reimbursement under policy ASJ-01, policyholder PH-1002.
> 
> Patient: Meena Sharma (synthetic). Admitted on 10 August 2026 to Lakeview Hospital
> (synthetic) with severe upper abdominal pain, diagnosed as acute pancreatitis and managed
> medically. Chose a single AC room at Rs 7,000 per day (nursing Rs 1,000 per day). Discharged on 14 August 2026.
> 
> Final hospital bill number LVH/26/11872. Total amount claimed: Rs 72,000.
> Discharge summary, final bill and investigation reports attached.

**Bill**

| Item | Category | Qty | Amount |
|---|---|---|---|
| Room charges (single AC) | room | 4 | ₹28,000 |
| Nursing charges | nursing | 4 | ₹4,000 |
| Consultant visits | professional | 8 | ₹12,000 |
| Medicines and IV fluids | pharmacy | 1 | ₹18,000 |
| Investigations (serum lipase, CT abdomen) | diagnostics | 1 | ₹9,600 |
| Gloves | consumables | 1 | ₹400 |
| **Total** |  |  | **₹72,000** |

**Agent trace** (from `logs/run_log.jsonl`)

| Time (UTC) | Agent | Step / tool | Result |
|---|---|---|---|
| 18:07:45.063 | workflow | start | intake |
| 18:07:45.065 | intake_agent | start | reading claim message and bill |
| 18:07:45.066 | intake_agent | extract_claim_fields | policy ASJ-01, claimed 72000, diagnosis: acute pancreatitis |
| 18:07:45.943 | intake_agent | reconcile_fields | groq (qwen/qwen3.8-27b): 10 fields agree · groq: 883/900 requests, 84,304/90,000 tokens left today |
| 18:07:45.944 | intake_agent | validate_policy | ASJ-01 found in policies.json |
| 18:07:45.945 | intake_agent | parse_bill | 6 line items, total 72000 |
| 18:07:45.975 | workflow | fan-out | Policy Retrieval and Fraud Signal start together |
| 18:07:45.986 | policy_agent | start | limits + clauses for ASJ-01 |
| 18:07:45.986 | fraud_agent | start | risk rules + prior-claim memory |
| 18:07:45.987 | policy_agent | get_policy_limits | SI 500000, room cap 5000/day, co-pay 5% |
| 18:07:45.987 | fraud_agent | lookup_prior_claims | 0 earlier claims by PH-1002 |
| 18:07:45.988 | fraud_agent | check_waiting_period | no waiting period applies |
| 18:07:45.990 | fraud_agent | score_risk | score 0 [] |
| 18:07:46.012 | policy_agent | search_policy_clauses | 'room rent boarding nursing expenses per day limit' -> ['ASJ-01:4.1.i', 'ASJ-01:3.45', 'ASJ-01:4.1.1.note2'] |
| 18:07:46.012 | cost_agent | start | 6 bill lines against ASJ-01 limits |
| 18:07:46.012 | policy_agent | search_policy_clauses | 'intensive care unit ICU expenses per day limit' -> ['ASJ-01:4.1.ii', 'ASJ-01:3.26', 'ASJ-01:3.25'] |
| 18:07:46.013 | cost_agent | classify_items | not payable: ['Gloves'] |
| 18:07:46.014 | policy_agent | search_policy_clauses | 'rates exceeding limits proportion other expenses' -> ['ASJ-01:4.1.1.note2', 'ASJ-01:5'] |
| 18:07:46.014 | cost_agent | room_rent_deduction | cuts: ['Room + nursing (4 days)', 'Consultant visits', 'Investigations (serum lipase, CT abdomen)'] |
| 18:07:46.015 | policy_agent | search_policy_clauses | 'co-payment admissible claim' -> ['ASJ-01:9.3', 'ASJ-01:3.11', 'ASJ-01:3.36.ii'] |
| 18:07:46.016 | cost_agent | compute_payable | payable 48925 of 72000; over-claim True |
| 18:07:46.017 | policy_agent | search_policy_clauses | 'first thirty days waiting period illness' -> ['ASJ-01:6.2.i', 'ASJ-01:3.19', 'ASJ-01:6.3'] |
| 18:07:46.019 | policy_agent | search_policy_clauses | 'pre-existing disease waiting period' -> ['ASJ-01:6.1', 'ASJ-01:3.42', 'ASJ-01:3.44'] |
| 18:07:46.020 | policy_agent | search_irdai_rules | 'items for which coverage is not available' -> ['IRDAI:List-I'] |
| 18:07:46.020 | policy_agent | search_irdai_rules | 'items subsumed into room charges' -> ['IRDAI:List-II'] |
| 18:07:46.022 | policy_agent | search_irdai_rules | 'subsumed into procedure charges' -> ['IRDAI:List-III'] |
| 18:07:46.023 | policy_agent | search_irdai_rules | 'items subsumed into costs of treatment' -> ['IRDAI:List-IV'] |
| 18:07:46.023 | policy_agent | done | 23 clauses above the similarity threshold |
| 18:07:46.034 | workflow | fan-in | policy, fraud and cost signals in; Manager decides |
| 18:07:46.044 | manager_agent | start | fan-in: policy, fraud and cost signals |
| 18:07:46.045 | manager_agent | apply_decision_rules | partial payout 48925 |
| 18:07:46.045 | manager_agent | verify_citations | cited ['ASJ-01:4.1.i', 'ASJ-01:4.1.1.note2', 'IRDAI:List-I', 'ASJ-01:9.3'] — all retrieved |
| 18:07:46.070 | manager_agent | write_decision | status closed |
| 18:07:46.071 | workflow | done | status closed |

**Decision:** `partial` · payout **₹48,925** of ₹72,000 claimed · confidence 0.87 · status `closed`  
**Rationale:** Partially approved: payable ₹48,925 of ₹72,000 claimed; deductions: Room + nursing (4 days) ₹12,000 (ASJ-01:4.1.i); Consultant visits ₹4,500 (ASJ-01:4.1.1.note2); Investigations (serum lipase, CT abdomen) ₹3,600 (ASJ-01:4.1.1.note2); Gloves ₹400 (IRDAI:List-I); Co-payment 5% ₹2,575 (ASJ-01:9.3).  
**Cited clauses:** `ASJ-01:4.1.i`, `ASJ-01:4.1.1.note2`, `IRDAI:List-I`, `ASJ-01:9.3`  
**Fraud score:** 0 

**Deduction table**

| Item | Cut | Reason | Clause |
|---|---|---|---|
| Room + nursing (4 days) | ₹12,000 | room rent, boarding and nursing above cap: 4 x (8000 - 5000) | `ASJ-01:4.1.i` |
| Consultant visits | ₹4,500 | proportionate deduction 5000/8000 | `ASJ-01:4.1.1.note2` |
| Investigations (serum lipase, CT abdomen) | ₹3,600 | proportionate deduction 5000/8000 | `ASJ-01:4.1.1.note2` |
| Gloves | ₹400 | non-payable item | `IRDAI:List-I` |
| Co-payment 5% | ₹2,575 | 5% co-payment on admissible claim | `ASJ-01:9.3` |

## Scenario 3 — reject: waiting period
Planned hernia repair six months into cover. Hernia is a specified condition with a 24-month wait (`ASJ-01:6.3.i`).

In [4]:
rec3 = await demo("CLM-0003")

### CLM-0003 — claim message

> Claim for reimbursement under policy ASJ-01, policyholder PH-1003.
> 
> Patient: Arjun Mehta (synthetic). Admitted on 14 July 2026 to Greenfield Hospital (synthetic)
> for a planned laparoscopic repair of a right inguinal hernia with mesh. Twin-sharing room at
> Rs 4,000 per day. Discharged on 16 July 2026.
> 
> Final hospital bill number GFH/26/03318. Total amount claimed: Rs 77,000.
> Discharge summary, final bill and investigation reports attached.

**Bill**

| Item | Category | Qty | Amount |
|---|---|---|---|
| Room charges (twin sharing) | room | 2 | ₹8,000 |
| Nursing charges | nursing | 2 | ₹2,000 |
| Surgeon fees - laparoscopic hernioplasty | professional | 1 | ₹28,000 |
| Anaesthetist fees | professional | 1 | ₹7,000 |
| Operation theatre charges | ot | 1 | ₹12,000 |
| Polypropylene mesh | implants | 1 | ₹9,000 |
| Medicines and IV fluids | pharmacy | 1 | ₹6,000 |
| Investigations (CBC, USG) | diagnostics | 1 | ₹5,000 |
| **Total** |  |  | **₹77,000** |

**Agent trace** (from `logs/run_log.jsonl`)

| Time (UTC) | Agent | Step / tool | Result |
|---|---|---|---|
| 18:07:56.859 | workflow | start | intake |
| 18:07:56.860 | intake_agent | start | reading claim message and bill |
| 18:07:56.862 | intake_agent | extract_claim_fields | policy ASJ-01, claimed 77000, diagnosis: planned laparoscopic repair of a right inguinal hernia with mesh |
| 18:07:57.739 | intake_agent | reconcile_fields | groq (qwen/qwen3.8-27b): 10 fields agree · groq: 882/900 requests, 83,859/90,000 tokens left today |
| 18:07:57.741 | intake_agent | validate_policy | ASJ-01 found in policies.json |
| 18:07:57.743 | intake_agent | parse_bill | 8 line items, total 77000 |
| 18:07:57.786 | workflow | fan-out | Policy Retrieval and Fraud Signal start together |
| 18:07:57.798 | policy_agent | start | limits + clauses for ASJ-01 |
| 18:07:57.798 | fraud_agent | start | risk rules + prior-claim memory |
| 18:07:57.799 | policy_agent | get_policy_limits | SI 500000, room cap 5000/day, co-pay 5% |
| 18:07:57.799 | fraud_agent | lookup_prior_claims | 0 earlier claims by PH-1003 |
| 18:07:57.801 | fraud_agent | check_waiting_period | excluded under ASJ-01:6.3.i until 2028-01-01 |
| 18:07:57.801 | fraud_agent | score_risk | score 0 [] |
| 18:07:57.828 | cost_agent | start | 8 bill lines against ASJ-01 limits |
| 18:07:57.829 | policy_agent | search_policy_clauses | 'room rent boarding nursing expenses per day limit' -> ['ASJ-01:4.1.i', 'ASJ-01:3.45', 'ASJ-01:4.1.1.note2'] |
| 18:07:57.830 | cost_agent | classify_items | not payable: none |
| 18:07:57.831 | policy_agent | search_policy_clauses | 'intensive care unit ICU expenses per day limit' -> ['ASJ-01:4.1.ii', 'ASJ-01:3.26', 'ASJ-01:3.25'] |
| 18:07:57.831 | cost_agent | room_rent_deduction | cuts: none |
| 18:07:57.832 | policy_agent | search_policy_clauses | 'rates exceeding limits proportion other expenses' -> ['ASJ-01:4.1.1.note2', 'ASJ-01:5'] |
| 18:07:57.833 | cost_agent | compute_payable | payable 73150 of 77000; over-claim False |
| 18:07:57.833 | policy_agent | search_policy_clauses | 'co-payment admissible claim' -> ['ASJ-01:9.3', 'ASJ-01:3.11', 'ASJ-01:3.36.ii'] |
| 18:07:57.835 | policy_agent | search_policy_clauses | 'first thirty days waiting period illness' -> ['ASJ-01:6.2.i', 'ASJ-01:3.19', 'ASJ-01:6.3'] |
| 18:07:57.838 | policy_agent | search_policy_clauses | 'pre-existing disease waiting period' -> ['ASJ-01:6.1', 'ASJ-01:3.42', 'ASJ-01:3.44'] |
| 18:07:57.838 | policy_agent | search_policy_clauses | 'waiting period hernia' -> ['ASJ-01:10.17.iii', 'ASJ-01:6.3.i', 'ASJ-01:6'] |
| 18:07:57.840 | policy_agent | search_irdai_rules | 'items for which coverage is not available' -> ['IRDAI:List-I'] |
| 18:07:57.842 | policy_agent | search_irdai_rules | 'items subsumed into room charges' -> ['IRDAI:List-II'] |
| 18:07:57.842 | policy_agent | search_irdai_rules | 'subsumed into procedure charges' -> ['IRDAI:List-III'] |
| 18:07:57.844 | policy_agent | search_irdai_rules | 'items subsumed into costs of treatment' -> ['IRDAI:List-IV'] |
| 18:07:57.844 | policy_agent | done | 28 clauses above the similarity threshold |
| 18:07:57.861 | workflow | fan-in | policy, fraud and cost signals in; Manager decides |
| 18:07:57.870 | manager_agent | start | fan-in: policy, fraud and cost signals |
| 18:07:57.870 | manager_agent | apply_decision_rules | reject payout 0 |
| 18:07:57.871 | manager_agent | verify_citations | cited ['ASJ-01:6.3.i'] — all retrieved |
| 18:07:57.904 | manager_agent | write_decision | status closed |
| 18:07:57.904 | workflow | done | status closed |

**Decision:** `reject` · payout **₹0** of ₹77,000 claimed · confidence 1.0 · status `closed`  
**Rationale:** Rejected: hernia of all types is inside its waiting period until 2028-01-01.  
**Cited clauses:** `ASJ-01:6.3.i`  
**Fraud score:** 0   
**Waiting period:** hernia of all types — excluded until 2028-01-01 (`ASJ-01:6.3.i`)

**Deduction table**

| Item | Cut | Reason | Clause |
|---|---|---|---|
| Co-payment 5% | ₹3,850 | 5% co-payment on admissible claim | `ASJ-01:9.3` |

## Scenario 4 — high value → human
Coronary angioplasty, ₹4,50,000 claimed (> ₹2,00,000). The Manager escalates with a recommended payout and the
Human Review Desk opens a ticket.

In [5]:
rec4 = await demo("CLM-0004")

### CLM-0004 — claim message

> Claim for reimbursement under policy ASJ-01, policyholder PH-1004.
> 
> Patient: Suresh Iyer (synthetic). Admitted on 5 October 2026 to Heartline Cardiac Centre
> (synthetic) with chest pain; diagnosed with acute coronary syndrome (triple-vessel disease) and
> underwent coronary angioplasty (PTCA) with two drug-eluting stents. Two days in ICU at Rs 10,000
> per day, then three days in a twin-sharing room at Rs 4,000 per day
> (nursing Rs 1,000 per day). Discharged on 10 October 2026.
> 
> Final hospital bill number HCC/26/00562. Total amount claimed: Rs 4,50,000.
> Discharge summary, final bill, angiography report and stent invoices attached.

**Bill**

| Item | Category | Qty | Amount |
|---|---|---|---|
| ICU charges | icu | 2 | ₹20,000 |
| Room charges (twin sharing) | room | 3 | ₹12,000 |
| Nursing charges (ward days) | nursing | 3 | ₹3,000 |
| Cardiologist fees - PTCA | professional | 1 | ₹90,000 |
| Cath lab charges | ot | 1 | ₹60,000 |
| Drug-eluting stents | implants | 2 | ₹1,60,000 |
| Medicines and consumables | pharmacy | 1 | ₹50,000 |
| Investigations (angiography, ECG, echo, labs) | diagnostics | 1 | ₹54,000 |
| Gloves | consumables | 1 | ₹600 |
| Mask | consumables | 1 | ₹400 |
| **Total** |  |  | **₹4,50,000** |

**Agent trace** (from `logs/run_log.jsonl`)

| Time (UTC) | Agent | Step / tool | Result |
|---|---|---|---|
| 18:08:06.674 | workflow | start | intake |
| 18:08:06.674 | intake_agent | start | reading claim message and bill |
| 18:08:06.674 | intake_agent | extract_claim_fields | policy ASJ-01, claimed 450000, diagnosis: acute coronary syndrome (triple-vessel disease); coronary angioplasty (PTCA) with two drug-eluting stents |
| 18:08:07.609 | intake_agent | reconcile_fields | groq (qwen/qwen3.8-27b): 10 fields agree · groq: 881/900 requests, 83,345/90,000 tokens left today |
| 18:08:07.610 | intake_agent | validate_policy | ASJ-01 found in policies.json |
| 18:08:07.611 | intake_agent | parse_bill | 10 line items, total 450000 |
| 18:08:07.648 | workflow | fan-out | Policy Retrieval and Fraud Signal start together |
| 18:08:07.658 | policy_agent | start | limits + clauses for ASJ-01 |
| 18:08:07.658 | fraud_agent | start | risk rules + prior-claim memory |
| 18:08:07.660 | policy_agent | get_policy_limits | SI 500000, room cap 5000/day, co-pay 5% |
| 18:08:07.660 | fraud_agent | lookup_prior_claims | 0 earlier claims by PH-1004 |
| 18:08:07.661 | fraud_agent | check_waiting_period | no waiting period applies |
| 18:08:07.661 | fraud_agent | score_risk | score 25 ['amount_near_sum_insured'] |
| 18:08:07.690 | policy_agent | search_policy_clauses | 'room rent boarding nursing expenses per day limit' -> ['ASJ-01:4.1.i', 'ASJ-01:3.45', 'ASJ-01:4.1.1.note2'] |
| 18:08:07.690 | cost_agent | start | 10 bill lines against ASJ-01 limits |
| 18:08:07.690 | policy_agent | search_policy_clauses | 'intensive care unit ICU expenses per day limit' -> ['ASJ-01:4.1.ii', 'ASJ-01:3.26', 'ASJ-01:3.25'] |
| 18:08:07.691 | cost_agent | classify_items | not payable: ['Gloves', 'Mask'] |
| 18:08:07.691 | policy_agent | search_policy_clauses | 'rates exceeding limits proportion other expenses' -> ['ASJ-01:4.1.1.note2', 'ASJ-01:5'] |
| 18:08:07.693 | cost_agent | room_rent_deduction | cuts: none |
| 18:08:07.693 | policy_agent | search_policy_clauses | 'co-payment admissible claim' -> ['ASJ-01:9.3', 'ASJ-01:3.11', 'ASJ-01:3.36.ii'] |
| 18:08:07.694 | cost_agent | compute_payable | payable 426550 of 450000; over-claim False |
| 18:08:07.694 | policy_agent | search_policy_clauses | 'first thirty days waiting period illness' -> ['ASJ-01:6.2.i', 'ASJ-01:3.19', 'ASJ-01:6.3'] |
| 18:08:07.697 | policy_agent | search_policy_clauses | 'pre-existing disease waiting period' -> ['ASJ-01:6.1', 'ASJ-01:3.42', 'ASJ-01:3.44'] |
| 18:08:07.697 | policy_agent | search_irdai_rules | 'items for which coverage is not available' -> ['IRDAI:List-I'] |
| 18:08:07.698 | policy_agent | search_irdai_rules | 'items subsumed into room charges' -> ['IRDAI:List-II'] |
| 18:08:07.699 | policy_agent | search_irdai_rules | 'subsumed into procedure charges' -> ['IRDAI:List-III'] |
| 18:08:07.700 | policy_agent | search_irdai_rules | 'items subsumed into costs of treatment' -> ['IRDAI:List-IV'] |
| 18:08:07.700 | policy_agent | done | 23 clauses above the similarity threshold |
| 18:08:07.712 | workflow | fan-in | policy, fraud and cost signals in; Manager decides |
| 18:08:07.721 | manager_agent | start | fan-in: policy, fraud and cost signals |
| 18:08:07.722 | manager_agent | apply_decision_rules | escalate payout 426550 ['high_value'] |
| 18:08:07.722 | manager_agent | verify_citations | cited ['IRDAI:List-I', 'ASJ-01:9.3'] — all retrieved |
| 18:08:07.736 | manager_agent | write_decision | status escalated |
| 18:08:07.751 | human_desk | create_review_ticket | ticket RV-0001 opened |
| 18:08:07.751 | human_desk | create_review_ticket | RV-0001 waiting for a reviewer |
| 18:08:07.761 | workflow | done | status escalated — waiting for a reviewer |

**Decision:** `escalate` · payout **₹4,26,550** of ₹4,50,000 claimed · confidence 1.0 · status `escalated`  
**Escalation:** high_value  
**Rationale:** Escalated to a human reviewer (claimed amount above the human-review threshold). Recommendation — Approved: payable ₹4,26,550 of ₹4,50,000 claimed; deductions: Gloves ₹600 (IRDAI:List-I); Mask ₹400 (IRDAI:List-I); Co-payment 5% ₹22,450 (ASJ-01:9.3).  
**Cited clauses:** `IRDAI:List-I`, `ASJ-01:9.3`  
**Fraud score:** 25 ['amount_near_sum_insured']

**Deduction table**

| Item | Cut | Reason | Clause |
|---|---|---|---|
| Gloves | ₹600 | non-payable item | `IRDAI:List-I` |
| Mask | ₹400 | non-payable item | `IRDAI:List-I` |
| Co-payment 5% | ₹22,450 | 5% co-payment on admissible claim | `ASJ-01:9.3` |

The reviewer checks the stent invoices and approves the recommendation; the verdict is written back and the claim closes.

In [6]:
rec4 = human_desk.review("CLM-0004", "approve",
                        "Stent invoices and angiography report match the bill; approve as recommended.",
                        reviewer="Demo Reviewer")
show_record(rec4)

**Decision:** `escalate` · payout **₹4,26,550** of ₹4,50,000 claimed · confidence 1.0 · status `closed`  
**Escalation:** high_value  
**Rationale:** Escalated to a human reviewer (claimed amount above the human-review threshold). Recommendation — Approved: payable ₹4,26,550 of ₹4,50,000 claimed; deductions: Gloves ₹600 (IRDAI:List-I); Mask ₹400 (IRDAI:List-I); Co-payment 5% ₹22,450 (ASJ-01:9.3).  
**Cited clauses:** `IRDAI:List-I`, `ASJ-01:9.3`  
**Fraud score:** 25 ['amount_near_sum_insured']

**Deduction table**

| Item | Cut | Reason | Clause |
|---|---|---|---|
| Gloves | ₹600 | non-payable item | `IRDAI:List-I` |
| Mask | ₹400 | non-payable item | `IRDAI:List-I` |
| Co-payment 5% | ₹22,450 | 5% co-payment on admissible claim | `ASJ-01:9.3` |

**Human review** RV-0001: `approve` by Demo Reviewer — payout ₹4,26,550. Note: Stent invoices and angiography report match the bill; approve as recommended.

## Scenarios 5–7
5 · fraud signal (admitted on day 20, bill number reused from a prior claim) · 6 · honest failure: the policy's
wording isn't in the corpus, so there is no grounding clause and the system refuses to decide · 7 · missing amount:
intake halts.

In [7]:
rec5 = await demo("CLM-0005")

### CLM-0005 — claim message

> Claim for reimbursement under policy ASJ-01, policyholder PH-1005.
> 
> Patient: Kavita Rao (synthetic). Injured in a road accident on 21 January 2026 and admitted the
> same day to City General Hospital (synthetic) with a fracture of the right distal radius;
> underwent open reduction and internal fixation with a plate. Twin-sharing room at Rs 4,000 per
> day. Discharged on 23 January 2026.
> 
> Final hospital bill number CGH/25/07731. Total amount claimed: Rs 99,300.
> Discharge summary, final bill and X-ray reports attached.

**Bill**

| Item | Category | Qty | Amount |
|---|---|---|---|
| Room charges (twin sharing) | room | 2 | ₹8,000 |
| Nursing charges | nursing | 2 | ₹1,600 |
| Orthopaedic surgeon fees - ORIF | professional | 1 | ₹30,000 |
| Anaesthetist fees | professional | 1 | ₹8,000 |
| Operation theatre charges | ot | 1 | ₹12,000 |
| Locking plate and screws | implants | 1 | ₹25,000 |
| Medicines and IV fluids | pharmacy | 1 | ₹9,000 |
| Investigations (X-ray, CT wrist) | diagnostics | 1 | ₹5,400 |
| Gloves | consumables | 1 | ₹300 |
| **Total** |  |  | **₹99,300** |

**Agent trace** (from `logs/run_log.jsonl`)

| Time (UTC) | Agent | Step / tool | Result |
|---|---|---|---|
| 18:08:24.859 | workflow | start | intake |
| 18:08:24.860 | intake_agent | start | reading claim message and bill |
| 18:08:24.861 | intake_agent | extract_claim_fields | policy ASJ-01, claimed 99300, diagnosis: fracture of the right distal radius; open reduction and internal fixation with a plate |
| 18:08:25.724 | intake_agent | reconcile_fields | groq (qwen/qwen3.8-27b): 10 fields agree · groq: 880/900 requests, 82,890/90,000 tokens left today |
| 18:08:25.725 | intake_agent | validate_policy | ASJ-01 found in policies.json |
| 18:08:25.726 | intake_agent | parse_bill | 9 line items, total 99300 |
| 18:08:25.758 | workflow | fan-out | Policy Retrieval and Fraud Signal start together |
| 18:08:25.770 | policy_agent | start | limits + clauses for ASJ-01 |
| 18:08:25.770 | fraud_agent | start | risk rules + prior-claim memory |
| 18:08:25.771 | policy_agent | get_policy_limits | SI 500000, room cap 5000/day, co-pay 5% |
| 18:08:25.771 | fraud_agent | lookup_prior_claims | 0 earlier claims by PH-1005 |
| 18:08:25.772 | fraud_agent | check_waiting_period | accident: waiting periods do not apply |
| 18:08:25.773 | fraud_agent | score_risk | score 75 ['soon_after_inception', 'duplicate_bill_number'] |
| 18:08:25.799 | policy_agent | search_policy_clauses | 'room rent boarding nursing expenses per day limit' -> ['ASJ-01:4.1.i', 'ASJ-01:3.45', 'ASJ-01:4.1.1.note2'] |
| 18:08:25.799 | cost_agent | start | 9 bill lines against ASJ-01 limits |
| 18:08:25.800 | policy_agent | search_policy_clauses | 'intensive care unit ICU expenses per day limit' -> ['ASJ-01:4.1.ii', 'ASJ-01:3.26', 'ASJ-01:3.25'] |
| 18:08:25.801 | cost_agent | classify_items | not payable: ['Gloves'] |
| 18:08:25.802 | policy_agent | search_policy_clauses | 'rates exceeding limits proportion other expenses' -> ['ASJ-01:4.1.1.note2', 'ASJ-01:5'] |
| 18:08:25.802 | cost_agent | room_rent_deduction | cuts: none |
| 18:08:25.803 | policy_agent | search_policy_clauses | 'co-payment admissible claim' -> ['ASJ-01:9.3', 'ASJ-01:3.11', 'ASJ-01:3.36.ii'] |
| 18:08:25.805 | cost_agent | compute_payable | payable 94050 of 99300; over-claim False |
| 18:08:25.806 | policy_agent | search_policy_clauses | 'first thirty days waiting period illness' -> ['ASJ-01:6.2.i', 'ASJ-01:3.19', 'ASJ-01:6.3'] |
| 18:08:25.807 | policy_agent | search_policy_clauses | 'pre-existing disease waiting period' -> ['ASJ-01:6.1', 'ASJ-01:3.42', 'ASJ-01:3.44'] |
| 18:08:25.808 | policy_agent | search_irdai_rules | 'items for which coverage is not available' -> ['IRDAI:List-I'] |
| 18:08:25.809 | policy_agent | search_irdai_rules | 'items subsumed into room charges' -> ['IRDAI:List-II'] |
| 18:08:25.809 | policy_agent | search_irdai_rules | 'subsumed into procedure charges' -> ['IRDAI:List-III'] |
| 18:08:25.811 | policy_agent | search_irdai_rules | 'items subsumed into costs of treatment' -> ['IRDAI:List-IV'] |
| 18:08:25.811 | policy_agent | done | 23 clauses above the similarity threshold |
| 18:08:25.824 | workflow | fan-in | policy, fraud and cost signals in; Manager decides |
| 18:08:25.835 | manager_agent | start | fan-in: policy, fraud and cost signals |
| 18:08:25.835 | manager_agent | apply_decision_rules | escalate payout 94050 ['fraud_risk'] |
| 18:08:25.836 | manager_agent | verify_citations | cited ['IRDAI:List-I', 'ASJ-01:9.3'] — all retrieved |
| 18:08:25.851 | manager_agent | write_decision | status escalated |
| 18:08:25.875 | human_desk | create_review_ticket | ticket RV-0002 opened |
| 18:08:25.876 | human_desk | create_review_ticket | RV-0002 waiting for a reviewer |
| 18:08:25.886 | workflow | done | status escalated — waiting for a reviewer |

**Decision:** `escalate` · payout **₹94,050** of ₹99,300 claimed · confidence 1.0 · status `escalated`  
**Escalation:** fraud_risk  
**Rationale:** Escalated to a human reviewer (fraud risk score at or above the threshold). Recommendation — Approved: payable ₹94,050 of ₹99,300 claimed; deductions: Gloves ₹300 (IRDAI:List-I); Co-payment 5% ₹4,950 (ASJ-01:9.3).  
**Cited clauses:** `IRDAI:List-I`, `ASJ-01:9.3`  
**Fraud score:** 75 ['soon_after_inception', 'duplicate_bill_number']

**Deduction table**

| Item | Cut | Reason | Clause |
|---|---|---|---|
| Gloves | ₹300 | non-payable item | `IRDAI:List-I` |
| Co-payment 5% | ₹4,950 | 5% co-payment on admissible claim | `ASJ-01:9.3` |

In [8]:
rec6 = await demo("CLM-0006")

### CLM-0006 — claim message

> Claim for reimbursement under policy GHC-01, policyholder PH-1006.
> 
> Patient: Neha Kapoor (synthetic). Admitted on 18 August 2026 to Riverside Hospital (synthetic)
> with high fever and low platelet count; diagnosed with dengue fever and treated medically.
> General ward at Rs 2,500 per day. Discharged on 21 August 2026.
> 
> Final hospital bill number RSH/26/09174. Total amount claimed: Rs 42,000.
> Discharge summary, final bill and lab reports attached.

**Bill**

| Item | Category | Qty | Amount |
|---|---|---|---|
| Room charges (general ward) | room | 3 | ₹7,500 |
| Nursing charges | nursing | 3 | ₹1,500 |
| Physician visits | professional | 6 | ₹6,000 |
| Medicines and IV fluids | pharmacy | 1 | ₹15,000 |
| Investigations (NS1, CBC x5) | diagnostics | 1 | ₹12,000 |
| **Total** |  |  | **₹42,000** |

**Agent trace** (from `logs/run_log.jsonl`)

| Time (UTC) | Agent | Step / tool | Result |
|---|---|---|---|
| 18:08:34.789 | workflow | start | intake |
| 18:08:34.790 | intake_agent | start | reading claim message and bill |
| 18:08:34.791 | intake_agent | extract_claim_fields | policy GHC-01, claimed 42000, diagnosis: dengue fever |
| 18:08:35.685 | intake_agent | reconcile_fields | groq (qwen/qwen3.8-27b): 10 fields agree · groq: 879/900 requests, 82,467/90,000 tokens left today |
| 18:08:35.686 | intake_agent | validate_policy | GHC-01 found in policies.json |
| 18:08:35.686 | intake_agent | parse_bill | 5 line items, total 42000 |
| 18:08:35.719 | workflow | fan-out | Policy Retrieval and Fraud Signal start together |
| 18:08:35.732 | policy_agent | start | limits + clauses for GHC-01 |
| 18:08:35.732 | fraud_agent | start | risk rules + prior-claim memory |
| 18:08:35.732 | policy_agent | get_policy_limits | SI 300000, room cap 3000/day, co-pay 0% |
| 18:08:35.732 | fraud_agent | lookup_prior_claims | 0 earlier claims by PH-1006 |
| 18:08:35.733 | fraud_agent | check_waiting_period | no waiting period applies |
| 18:08:35.734 | fraud_agent | score_risk | score 0 [] |
| 18:08:35.756 | cost_agent | start | 5 bill lines against GHC-01 limits |
| 18:08:35.757 | cost_agent | classify_items | not payable: none |
| 18:08:35.759 | cost_agent | room_rent_deduction | cuts: none |
| 18:08:35.760 | cost_agent | compute_payable | payable 42000 of 42000; over-claim False |
| 18:08:35.760 | policy_agent | search_policy_clauses | 'room rent boarding nursing expenses per day limit' -> [] |
| 18:08:35.761 | policy_agent | search_policy_clauses | 'intensive care unit ICU expenses per day limit' -> [] |
| 18:08:35.761 | policy_agent | search_policy_clauses | 'rates exceeding limits proportion other expenses' -> [] |
| 18:08:35.762 | policy_agent | search_policy_clauses | 'co-payment admissible claim' -> [] |
| 18:08:35.762 | policy_agent | search_policy_clauses | 'first thirty days waiting period illness' -> [] |
| 18:08:35.764 | policy_agent | search_policy_clauses | 'pre-existing disease waiting period' -> [] |
| 18:08:35.765 | policy_agent | search_irdai_rules | 'items for which coverage is not available' -> ['IRDAI:List-I'] |
| 18:08:35.765 | policy_agent | search_irdai_rules | 'items subsumed into room charges' -> ['IRDAI:List-II'] |
| 18:08:35.766 | policy_agent | search_irdai_rules | 'subsumed into procedure charges' -> ['IRDAI:List-III'] |
| 18:08:35.766 | policy_agent | search_irdai_rules | 'items subsumed into costs of treatment' -> ['IRDAI:List-IV'] |
| 18:08:35.767 | policy_agent | done | 4 clauses above the similarity threshold |
| 18:08:35.780 | workflow | fan-in | policy, fraud and cost signals in; Manager decides |
| 18:08:35.788 | manager_agent | start | fan-in: policy, fraud and cost signals |
| 18:08:35.789 | manager_agent | apply_decision_rules | escalate payout 42000 ['no_grounding'] |
| 18:08:35.790 | manager_agent | verify_citations | cited [] — all retrieved |
| 18:08:35.801 | manager_agent | write_decision | status escalated |
| 18:08:35.826 | human_desk | create_review_ticket | ticket RV-0003 opened |
| 18:08:35.826 | human_desk | create_review_ticket | RV-0003 waiting for a reviewer |
| 18:08:35.834 | workflow | done | status escalated — waiting for a reviewer |

**Decision:** `escalate` · payout **₹42,000** of ₹42,000 claimed · confidence 0.0 · status `escalated`  
**Escalation:** no_grounding  
**Rationale:** Escalated to a human reviewer (no grounding clause found). Recommendation — Approved: payable ₹42,000 of ₹42,000 claimed.  
**Cited clauses:** — none  
**Fraud score:** 0 

In [9]:
rec7 = await demo("CLM-0007")

### CLM-0007 — claim message

> Claim for reimbursement under policy ASJ-01, policyholder PH-1007.
> 
> Patient: Imran Sheikh (synthetic). Admitted on 2 October 2026 to Lakeview Hospital (synthetic)
> with acute gastroenteritis and dehydration; treated with IV fluids. Discharged on 4 October 2026.
> 
> The hospital has not yet issued the final bill; I will send it separately.

**Bill:** none attached

**Agent trace** (from `logs/run_log.jsonl`)

| Time (UTC) | Agent | Step / tool | Result |
|---|---|---|---|
| 18:08:40.754 | workflow | start | intake |
| 18:08:40.755 | intake_agent | start | reading claim message and bill |
| 18:08:40.755 | intake_agent | halt | claimed amount is missing — ask the claimant for the amount claimed and the final bill |
| 18:08:40.769 | intake_agent | set_halt | halted: claimed amount is missing — ask the claimant for the amount claimed and the final bill |
| 18:08:40.772 | workflow | stop | intake halted: missing_claimed_amount |

**Intake halted** — `missing_claimed_amount`: claimed amount is missing — ask the claimant for the amount claimed and the final bill  
Status: `intake` · no downstream agent ran.

## All seven scenarios — expected vs actual
Expected values are the hand-worked `data/claims/<id>/expected.json`.

In [10]:
finals = {"CLM-0001": rec1, "CLM-0002": rec2, "CLM-0003": rec3, "CLM-0004": rec4,
          "CLM-0005": rec5, "CLM-0006": rec6, "CLM-0007": rec7}
rows, passed = [], 0
for cid, rec in finals.items():
    exp = ExpectedOutcome.model_validate(json.loads((CLAIMS / cid / "expected.json").read_text(encoding="utf-8")))
    s = summary(rec)
    exp_verdict = exp.verdict or "halt"
    ok = (s["verdict"] == exp_verdict and s["status"] == exp.final_status
          and (exp.halt is not None or s["payout"] == exp.payable)
          and s["escalation"] == exp.escalation_reasons)
    passed += ok
    rows.append((exp.scenario, cid, exp_verdict, s["verdict"], rupees(exp.payable), rupees(s["payout"]),
                 ", ".join(s["escalation"]) or "—", s["status"], "✅" if ok else "❌"))
display(Markdown(table(["#", "Claim", "Expected", "Actual", "Expected ₹", "Actual ₹", "Escalation", "Status", "Match"], rows)))
print(f"{passed}/7 scenarios match.  {DISCLAIMER}")

| # | Claim | Expected | Actual | Expected ₹ | Actual ₹ | Escalation | Status | Match |
|---|---|---|---|---|---|---|---|---|
| 1 | CLM-0001 | approve | approve | ₹79,610 | ₹79,610 | — | closed | ✅ |
| 2 | CLM-0002 | partial | partial | ₹48,925 | ₹48,925 | — | closed | ✅ |
| 3 | CLM-0003 | reject | reject | ₹0 | ₹0 | — | closed | ✅ |
| 4 | CLM-0004 | escalate | escalate | ₹4,26,550 | ₹4,26,550 | high_value | closed | ✅ |
| 5 | CLM-0005 | escalate | escalate | ₹94,050 | ₹94,050 | fraud_risk | escalated | ✅ |
| 6 | CLM-0006 | escalate | escalate | ₹42,000 | ₹42,000 | no_grounding | escalated | ✅ |
| 7 | CLM-0007 | halt | halt | — | — | — | intake | ✅ |

7/7 scenarios match.  Educational prototype — not insurance, legal or medical advice.
